# 01 — Explore sources

Read-only scan of the four `IngestSource`s from Phase A. Use this to:

- Sanity-check that source DBs are reachable and contain the expected shapes
- Get item counts per source (informs `--limit` choices for eval runs)
- Eyeball content lengths to inform chunker / chunk_size decisions
- Sample a handful of items per source for hand-curation in notebook 02

In [ ]:
import os
from pathlib import Path

import pandas as pd

from domains.research.sources import ResearchSource
from domains.sessions.sources import SessionsSource
from domains.wiki.sources import LocalFileSource, RawStoreSource

BACKUP = Path(os.environ.get("BACKUP_SOURCE_DIR", "~")).expanduser()
BACKUP

In [ ]:
# Build sources. Skip any whose path is missing.
sources = {}
if (p := BACKUP / "raw_store.db").exists():
    sources["raw_store"] = RawStoreSource(p)
if (p := BACKUP / "sessions.db").exists():
    sources["sessions"] = SessionsSource(p)
if (p := BACKUP / "research.db").exists():
    sources["research"] = ResearchSource(p)
if (p := BACKUP / "notes").is_dir():
    sources["notes"] = LocalFileSource(p)
list(sources)

In [ ]:
# Item counts per source.
counts = {name: len(s.get_item_ids()) for name, s in sources.items()}
pd.Series(counts, name="n_items")

In [ ]:
# Content-length distribution per source — chars per item. Informs chunk_size choice.
frames = []
for name, s in sources.items():
    items = s.get_items()
    frames.append(
        pd.DataFrame(
            {"source": name, "chars": [len(item.text) for item in items]}
        )
    )
lengths = pd.concat(frames, ignore_index=True)
lengths.groupby("source")["chars"].describe()

In [ ]:
# Sample 3 items per source for visual inspection.
for name, s in sources.items():
    print(f"\n=== {name} ===")
    for item in s.get_items()[:3]:
        print(f"  {item.item_id}: {item.title!r} ({len(item.text)} chars)")